# Extracción automática de notas de corte desde PDF a CSV

Este cuaderno extrae de forma automática la información del PDF de notas de corte del curso **2025/2026** y genera un fichero CSV con estas columnas exactas:

1. **Nombre de la Universidad**
2. **Rama de estudios**
3. **Nombre de la titulación**
4. **Nota de corte ordinaria del grupo 1**

## Enfoque utilizado

Se aplica una estrategia **híbrida y conservadora**:

- **Extracción principal por tablas** con `pdfplumber`, porque el PDF tiene una estructura tabular muy marcada.
- **Parser de respaldo basado en texto y contexto** por si alguna página o fila no pudiera reconstruirse bien desde tablas.
- **Normalización y validación final** para evitar registros vacíos, notas mal formadas o duplicados exactos en el esquema final.

## Ficheros

- **Entrada**: `/mnt/data/notascortedum25-26.pdf`
- **Salida**: `/mnt/data/notas_corte_grupo1_ordinaria.csv`

> Criterio de calidad: ante una fila dudosa, el parser prefiere **omitirla** antes que inventar datos.


In [1]:
# Instalación/importación de librerías
# Si alguna librería no estuviera disponible, descomenta la línea correspondiente.

# %pip install pdfplumber pandas

import os
import re
import json
from typing import List, Dict, Optional, Tuple

import pandas as pd
import pdfplumber

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 50)


In [2]:
# Configuración de rutas

PDF_PATH = r"/mnt/data/notascortedum25-26.pdf"
CSV_PATH = r"/mnt/data/notas_corte_grupo1_ordinaria.csv"

assert os.path.exists(PDF_PATH), f"No se encuentra el PDF en la ruta esperada: {PDF_PATH}"

print("PDF de entrada:", PDF_PATH)
print("CSV de salida:", CSV_PATH)


PDF de entrada: /mnt/data/notascortedum25-26.pdf
CSV de salida: /mnt/data/notas_corte_grupo1_ordinaria.csv


In [3]:
# Extracción del PDF
# En esta celda se definen las funciones auxiliares y el parser principal.

UNIVERSIDADES_MAP = {
    "UNIVERSIDAD COMPLUTENSE DE MADRID": "Universidad Complutense de Madrid",
    "UNIVERSIDAD DE ALCALÁ": "Universidad de Alcalá",
    "UNIVERSIDAD AUTÓNOMA DE MADRID": "Universidad Autónoma de Madrid",
    "UNIVERSIDAD CARLOS III DE MADRID": "Universidad Carlos III de Madrid",
    "UNIVERSIDAD POLITÉCNICA DE MADRID": "Universidad Politécnica de Madrid",
    "UNIVERSIDAD REY JUAN CARLOS": "Universidad Rey Juan Carlos",
}

RAMAS_CLASICAS = [
    "Artes y Humanidades",
    "Ciencias",
    "Ciencias de la Salud",
    "Ciencias Sociales y Jurídicas",
    "Ingeniería y Arquitectura",
]

RAMAS_ESPECIALES = [
    "Dobles Grados",
    "Grados Interuniversitarios",
    "Grados conjuntos",
    "Centros adscritos",
]

# Mapa de normalización para aceptar variantes de mayúsculas/minúsculas.
RAMAS_MAP = {rama.lower(): rama for rama in RAMAS_CLASICAS + RAMAS_ESPECIALES}
RAMAS_MAP["grados conjuntos"] = "Grados conjuntos"
RAMAS_MAP["centros adscritos"] = "Centros adscritos"

PATRON_CODIGO = re.compile(r"^\d{4}$")
PATRON_NOTA_BRUTA = re.compile(r"^\d{1,2}[,.]\d{2,3}(?:\s*\(\d+\))?$")
PATRON_NOTA_LIMPIA = re.compile(r"^\d{1,2},\d{2,3}$")

def normalizar_texto(texto: Optional[str]) -> str:
    """Limpia espacios, saltos de línea y pequeñas anomalías tipográficas."""
    if texto is None:
        return ""
    texto = str(texto).replace("\xa0", " ").replace("\r", " ").replace("\t", " ")
    texto = texto.replace("—", "-")
    texto = texto.replace("HIstoria", "Historia")  # pequeña anomalía observada
    texto = re.sub(r"\s+", " ", texto.replace("\n", " ")).strip()
    return texto

def detectar_universidad(texto: str, universidad_actual: Optional[str] = None) -> Optional[str]:
    """Detecta la universidad activa a partir del encabezado de página o de bloque."""
    texto_may = normalizar_texto(texto).upper()
    for patron, nombre in UNIVERSIDADES_MAP.items():
        if patron in texto_may:
            return nombre
    return universidad_actual

def detectar_rama(texto: str, rama_actual: Optional[str] = None) -> Optional[str]:
    """Detecta la rama o bloque especial visible en un texto dado."""
    texto_norm = normalizar_texto(texto).lower()
    for patron in sorted(RAMAS_MAP.keys(), key=len, reverse=True):
        if patron in texto_norm:
            return RAMAS_MAP[patron]
    return rama_actual

def limpiar_nota(nota: Optional[str]) -> Optional[str]:
    """Conserva solo la nota, eliminando anotaciones del tipo '(11)'.

    Ejemplos:
    - '8,525 (11)' -> '8,525'
    - '-----' -> None
    - '8.65' -> '8,65'
    """
    nota = normalizar_texto(nota)
    if not nota or nota in {"-----", "---", "-", ""}:
        return None

    match = re.match(r"^(\d{1,2}[,.]\d{2,3})", nota)
    if not match:
        return None

    nota_limpia = match.group(1).replace(".", ",")
    return nota_limpia

def es_fila_de_titulacion(row: List[Optional[str]]) -> bool:
    """Determina si una fila de tabla parece una fila real de datos."""
    if not row:
        return False

    codigo = normalizar_texto(row[0] if len(row) > 0 else "")
    titulacion = normalizar_texto(row[1] if len(row) > 1 else "")
    nota_g1_ordinaria = limpiar_nota(row[2] if len(row) > 2 else "")

    return bool(PATRON_CODIGO.match(codigo) and titulacion and nota_g1_ordinaria)

def parsear_fila_titulacion(
    row: List[Optional[str]],
    universidad: Optional[str],
    rama: Optional[str],
    pagina: int,
) -> Optional[Dict[str, str]]:
    """Transforma una fila tabular en un registro final, si cumple los mínimos."""
    if not es_fila_de_titulacion(row):
        return None

    codigo = normalizar_texto(row[0])
    titulacion = normalizar_texto(row[1])
    nota_g1_ordinaria = limpiar_nota(row[2] if len(row) > 2 else "")

    if not codigo or not titulacion or not nota_g1_ordinaria:
        return None

    return {
        "Nombre de la Universidad": universidad,
        "Rama de estudios": rama,
        "Nombre de la titulación": titulacion,
        "Nota de corte ordinaria del grupo 1": nota_g1_ordinaria,
        "_codigo": codigo,
        "_pagina": pagina,
        "_metodo_extraccion": "tablas",
    }

def obtener_texto_superior(page, bbox: Tuple[float, float, float, float], margen: int = 40) -> str:
    """Extrae una pequeña franja de texto por encima de la tabla para detectar encabezados."""
    x0, top, x1, bottom = bbox
    franja = page.crop((0, max(0, top - margen), page.width, top))
    return normalizar_texto(franja.extract_text() or "")

def es_tabla_cabecera(table: List[List[Optional[str]]]) -> bool:
    """Identifica tablas compuestas solo por la cabecera de columnas."""
    if not table or not table[0]:
        return False
    primera = normalizar_texto(table[0][0] if len(table[0]) > 0 else "")
    return primera.lower() == "código"

def extraer_registros_por_texto(
    page_text: str,
    universidad: Optional[str],
    rama_inicial: Optional[str] = None,
    pagina: int = 0,
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    """Parser de respaldo basado en estados/contexto sobre texto plano.

    Solo se usa si la extracción por tablas no devuelve registros para una página.
    """
    registros = []
    lineas_problematicas = []
    rama_actual = rama_inicial
    buffer = []

    for linea in (page_text or "").splitlines():
        linea = normalizar_texto(linea)
        if not linea:
            continue

        nueva_uni = detectar_universidad(linea, universidad)
        if nueva_uni != universidad:
            universidad = nueva_uni
            continue

        # Si la línea es un encabezado de rama, actualizamos contexto y seguimos.
        nueva_rama = detectar_rama(linea, rama_actual)
        if nueva_rama != rama_actual and linea.lower() in RAMAS_MAP:
            rama_actual = nueva_rama
            continue

        # Líneas de cabecera/leyenda/pie que hay que ignorar.
        if any(
            patron.lower() in linea.lower()
            for patron in [
                "curso 2025/2026",
                "código titulaciones de grado",
                "grupo 1 pruebas de acceso",
                "grupo 2 titulados",
                "grupo 3 mayores",
                "grupo 4 mayores",
                "grupo 5 mayores",
                "grupo 6 bachillerato",
                "nota de corte correspondiente",
                "estudios sin plazas",
                "ordinaria extraord.",
                "ord. ext.",
            ]
        ):
            continue

        # Si empieza por código, iniciamos o cerramos buffer de registro.
        if PATRON_CODIGO.match(linea[:4]):
            if buffer:
                texto_fila = " ".join(buffer)
                match = re.match(
                    r"^(\d{4})\s+(.+?)\s+(\d{1,2}[,.]\d{2,3}(?:\s*\(\d+\))?|-----|---)\b",
                    texto_fila,
                )
                if match:
                    nota = limpiar_nota(match.group(3))
                    if nota:
                        registros.append(
                            {
                                "Nombre de la Universidad": universidad,
                                "Rama de estudios": rama_actual,
                                "Nombre de la titulación": normalizar_texto(match.group(2)),
                                "Nota de corte ordinaria del grupo 1": nota,
                                "_codigo": match.group(1),
                                "_pagina": pagina,
                                "_metodo_extraccion": "texto",
                            }
                        )
                else:
                    lineas_problematicas.append(
                        {
                            "pagina": pagina,
                            "universidad": universidad,
                            "rama": rama_actual,
                            "origen": "texto",
                            "contenido": texto_fila,
                        }
                    )
            buffer = [linea]
        else:
            if buffer:
                buffer.append(linea)

    # Vaciado final del buffer
    if buffer:
        texto_fila = " ".join(buffer)
        match = re.match(
            r"^(\d{4})\s+(.+?)\s+(\d{1,2}[,.]\d{2,3}(?:\s*\(\d+\))?|-----|---)\b",
            texto_fila,
        )
        if match:
            nota = limpiar_nota(match.group(3))
            if nota:
                registros.append(
                    {
                        "Nombre de la Universidad": universidad,
                        "Rama de estudios": rama_actual,
                        "Nombre de la titulación": normalizar_texto(match.group(2)),
                        "Nota de corte ordinaria del grupo 1": nota,
                        "_codigo": match.group(1),
                        "_pagina": pagina,
                        "_metodo_extraccion": "texto",
                    }
                )
        else:
            lineas_problematicas.append(
                {
                    "pagina": pagina,
                    "universidad": universidad,
                    "rama": rama_actual,
                    "origen": "texto",
                    "contenido": texto_fila,
                }
            )

    return registros, lineas_problematicas

def extraer_registros_pdf(pdf_path: str) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    """Función principal de extracción."""
    registros = []
    lineas_problematicas = []
    universidad_actual = None

    with pdfplumber.open(pdf_path) as pdf:
        for numero_pagina, page in enumerate(pdf.pages, start=1):
            texto_pagina = page.extract_text() or ""
            universidad_actual = detectar_universidad(texto_pagina, universidad_actual)

            tablas_encontradas = page.find_tables()
            rama_actual = None
            registros_tablas_pagina = []

            for table_obj in tablas_encontradas:
                table = table_obj.extract()
                texto_superior = obtener_texto_superior(page, table_obj.bbox, margen=40)

                # Intentamos actualizar el contexto de rama a partir del texto que hay justo encima.
                rama_actual = detectar_rama(texto_superior, rama_actual)

                # Si es solo cabecera, nos sirve para contexto pero no aporta datos.
                if es_tabla_cabecera(table):
                    rama_actual = detectar_rama(texto_superior, rama_actual)
                    continue

                # Si la primera fila de la tabla es una rama visible, la usamos como encabezado.
                if table and table[0]:
                    primera_fila_no_vacia = [normalizar_texto(c) for c in table[0] if normalizar_texto(c)]
                    primera_celda = primera_fila_no_vacia[0] if primera_fila_no_vacia else ""

                    if primera_celda.lower() in RAMAS_MAP:
                        rama_actual = RAMAS_MAP[primera_celda.lower()]
                        filas_datos = table[1:]
                    else:
                        filas_datos = table
                else:
                    filas_datos = []

                # Si todavía no tenemos rama, intentamos extraerla del texto completo de la página.
                if rama_actual is None:
                    rama_actual = detectar_rama(texto_pagina, rama_actual)

                for row in filas_datos:
                    registro = parsear_fila_titulacion(
                        row=row,
                        universidad=universidad_actual,
                        rama=rama_actual,
                        pagina=numero_pagina,
                    )

                    if registro:
                        registros.append(registro)
                        registros_tablas_pagina.append(registro)
                    else:
                        fila_texto = [normalizar_texto(c) for c in row if normalizar_texto(c)]
                        # Guardamos para depuración solo las filas que no son un simple encabezado textual.
                        if fila_texto and not (len(fila_texto) == 1 and not PATRON_CODIGO.match(fila_texto[0])):
                            lineas_problematicas.append(
                                {
                                    "pagina": numero_pagina,
                                    "universidad": universidad_actual,
                                    "rama": rama_actual,
                                    "origen": "tabla",
                                    "contenido": fila_texto,
                                }
                            )

            # Respaldo: si en una página no se ha extraído nada por tablas, probamos parser textual.
            if not registros_tablas_pagina:
                registros_texto, problemas_texto = extraer_registros_por_texto(
                    page_text=texto_pagina,
                    universidad=universidad_actual,
                    rama_inicial=rama_actual,
                    pagina=numero_pagina,
                )
                registros.extend(registros_texto)
                lineas_problematicas.extend(problemas_texto)

    return registros, lineas_problematicas

registros_extraidos, lineas_problematicas = extraer_registros_pdf(PDF_PATH)

print(f"Registros brutos extraídos: {len(registros_extraidos)}")
print(f"Entradas problemáticas guardadas para depuración: {len(lineas_problematicas)}")


Registros brutos extraídos: 602
Entradas problemáticas guardadas para depuración: 0


In [4]:
# Limpieza y normalización

# Convertimos a DataFrame.
df_bruto = pd.DataFrame(registros_extraidos)

if df_bruto.empty:
    raise ValueError("No se ha podido extraer ningún registro del PDF.")

# Nos quedamos con las columnas de trabajo y eliminamos duplicados exactos
# dentro del esquema final solicitado.
columnas_finales = [
    "Nombre de la Universidad",
    "Rama de estudios",
    "Nombre de la titulación",
    "Nota de corte ordinaria del grupo 1",
]

columnas_auxiliares = ["_codigo", "_pagina", "_metodo_extraccion"]

df_trabajo = df_bruto[columnas_finales + columnas_auxiliares].copy()

# Normalización extra por seguridad.
for col in columnas_finales:
    df_trabajo[col] = df_trabajo[col].astype(str).map(normalizar_texto)

df_trabajo["Nota de corte ordinaria del grupo 1"] = (
    df_trabajo["Nota de corte ordinaria del grupo 1"].map(limpiar_nota)
)

# Eliminamos filas inválidas o incompletas.
df_trabajo = df_trabajo[
    df_trabajo["Nombre de la Universidad"].ne("")
    & df_trabajo["Nombre de la titulación"].ne("")
    & df_trabajo["Nota de corte ordinaria del grupo 1"].notna()
].copy()

# Eliminamos duplicados exactos según el esquema final solicitado.
# Esto es importante sobre todo en algunos centros adscritos donde,
# al no incluirse el centro como columna, puede haber filas idénticas.
df_trabajo = df_trabajo.drop_duplicates(subset=columnas_finales).reset_index(drop=True)

print("Filas tras limpieza y deduplicación:", len(df_trabajo))


Filas tras limpieza y deduplicación: 594


In [5]:
# Construcción del DataFrame final

df_final = df_trabajo[columnas_finales].copy()

# Orden opcional para dejar una salida más estable y cómoda de revisar.
df_final = df_final.sort_values(
    by=["Nombre de la Universidad", "Rama de estudios", "Nombre de la titulación"],
    kind="stable"
).reset_index(drop=True)

print("Columnas finales:")
print(list(df_final.columns))
print()
print("Número total de filas extraídas:", len(df_final))


Columnas finales:
['Nombre de la Universidad', 'Rama de estudios', 'Nombre de la titulación', 'Nota de corte ordinaria del grupo 1']

Número total de filas extraídas: 594


In [6]:
# Exportación a CSV

# Se exporta con separador ';' para respetar bien las comas decimales españolas.
df_final.to_csv(CSV_PATH, index=False, sep=";", encoding="utf-8")

print("CSV generado correctamente en:")
print(CSV_PATH)


CSV generado correctamente en:
/mnt/data/notas_corte_grupo1_ordinaria.csv


In [7]:
# Vista previa y validaciones

print("Número total de filas extraídas:", len(df_final))
print()

print("Primeras 20 filas:")
display(df_final.head(20))

print("\nListado único de universidades detectadas:")
universidades_detectadas = sorted(df_final["Nombre de la Universidad"].dropna().unique().tolist())
for u in universidades_detectadas:
    print("-", u)

print("\nListado único de ramas detectadas:")
ramas_detectadas = sorted(df_final["Rama de estudios"].dropna().unique().tolist())
for r in ramas_detectadas:
    print("-", r)

# Validaciones obligatorias
filas_sin_universidad = df_final[df_final["Nombre de la Universidad"].eq("")]
filas_sin_titulacion = df_final[df_final["Nombre de la titulación"].eq("")]
filas_nota_sospechosa = df_final[
    ~df_final["Nota de corte ordinaria del grupo 1"].astype(str).str.match(PATRON_NOTA_LIMPIA, na=False)
]

print("\nValidación 1 - Filas sin universidad:", len(filas_sin_universidad))
print("Validación 2 - Filas sin titulación:", len(filas_sin_titulacion))
print("Validación 3 - Filas con nota sospechosa:", len(filas_nota_sospechosa))

if not filas_nota_sospechosa.empty:
    print("\nEjemplos de filas con nota sospechosa:")
    display(filas_nota_sospechosa.head(20))
else:
    print("\nNo se han detectado notas sospechosas con el patrón exigido.")


Número total de filas extraídas: 594

Primeras 20 filas:


,Nombre de la Universidad,Rama de estudios,Nombre de la titulación,Nota de corte ordinaria del grupo 1
0,Universidad Autónoma de Madrid,Artes y Humanidades,Arqueología,"5,000"
1,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios Clásicos y de la Antigüedad,"5,000"
2,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios Hispánicos: Lengua Española y sus Literaturas,"5,000"
3,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios Ingleses,"5,000"
4,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios de Asia y África (Chino),"5,000"
5,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios de Asia y África (Japonés),"5,000"
6,Universidad Autónoma de Madrid,Artes y Humanidades,Estudios de Asia y África (Árabe),"5,000"
7,Universidad Autónoma de Madrid,Artes y Humanidades,Filosofía,"5,000"
8,Universidad Autónoma de Madrid,Artes y Humanidades,Historia,"5,000"
9,Universidad Autónoma de Madrid,Artes y Humanidades,Historia del Arte,"5,000"



Listado único de universidades detectadas:
- Universidad Autónoma de Madrid
- Universidad Carlos III de Madrid
- Universidad Complutense de Madrid
- Universidad Politécnica de Madrid
- Universidad Rey Juan Carlos
- Universidad de Alcalá

Listado único de ramas detectadas:
- Artes y Humanidades
- Centros adscritos
- Ciencias
- Ciencias Sociales y Jurídicas
- Ciencias de la Salud
- Dobles Grados
- Grados Interuniversitarios
- Grados conjuntos
- Ingeniería y Arquitectura

Validación 1 - Filas sin universidad: 0
Validación 2 - Filas sin titulación: 0
Validación 3 - Filas con nota sospechosa: 0

No se han detectado notas sospechosas con el patrón exigido.


In [8]:
# Resumen de calidad de datos y depuración

resumen_calidad = {
    "filas_finales": int(len(df_final)),
    "universidades_unicas": int(df_final["Nombre de la Universidad"].nunique()),
    "ramas_unicas": int(df_final["Rama de estudios"].nunique()),
    "filas_sin_universidad": int(df_final["Nombre de la Universidad"].eq("").sum()),
    "filas_sin_titulacion": int(df_final["Nombre de la titulación"].eq("").sum()),
    "filas_con_nota_sospechosa": int(
        (~df_final["Nota de corte ordinaria del grupo 1"].astype(str).str.match(PATRON_NOTA_LIMPIA, na=False)).sum()
    ),
    "filas_problematicas_guardadas": int(len(lineas_problematicas)),
}

print("Resumen de calidad de datos:")
print(json.dumps(resumen_calidad, ensure_ascii=False, indent=2))

print("\nMuestra de entradas problemáticas guardadas para revisar posibles mejoras del parser:")
for item in lineas_problematicas[:20]:
    print(json.dumps(item, ensure_ascii=False))

# Comprobación final de existencia del CSV
assert os.path.exists(CSV_PATH), "El CSV no se ha generado correctamente."

print("\nProceso completado.")


Resumen de calidad de datos:
{
  "filas_finales": 594,
  "universidades_unicas": 6,
  "ramas_unicas": 9,
  "filas_sin_universidad": 0,
  "filas_sin_titulacion": 0,
  "filas_con_nota_sospechosa": 0,
  "filas_problematicas_guardadas": 0
}

Muestra de entradas problemáticas guardadas para revisar posibles mejoras del parser:



Proceso completado.
